# Chapter 4 — Cholesky Decomposition for Correlated Data Synthesis

Cholesky decomposition factorises a positive-definite covariance matrix **Σ = L Lᵀ**, where **L** is a lower-triangular matrix. Multiplying uncorrelated standard-normal samples by **Lᵀ** injects the desired correlation structure, yielding synthetic variables that match both the target marginal distributions and the inter-variable correlations.

**When to use this approach:**
- Marginal distributions for every variable are known (or can be fitted from a small representative sample).
- A correlation matrix between variable pairs is known or can be assumed.
- You need a *deterministic*, algebraically transparent synthesis algorithm (no black-box components).

**Notebook structure:**
1. Core concept — 2-variable example
2. Generalising to N variables
3. Parametric case — user-defined correlation matrix
4. Full pipeline — distribution fitting + KS-test validation

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from fitter import Fitter

## Section 1 — Core Concept: 2-Variable Example

The key function below:
1. Computes the Cholesky factor **L** of the covariance matrix.
2. Draws independent standard-normal samples.
3. Left-multiplies by **Lᵀ** and shifts by the mean vector.

The result is a sample from the multivariate normal with the specified mean and covariance.

In [ ]:
def generate_correlated_variables(mean, covariance, num_samples):
    """Generate correlated random variables via Cholesky decomposition."""
    L = np.linalg.cholesky(covariance)
    z = np.random.normal(size=(num_samples, len(mean)))
    return mean + z @ L.T

In [ ]:
# Target: two variables with correlation 0.7
mean_2d = [0, 0]
cov_2d  = np.array([[1.0, 0.7],
                    [0.7, 1.0]])

samples_2d = generate_correlated_variables(mean_2d, cov_2d, num_samples=1000)
df2 = pd.DataFrame(samples_2d, columns=['X1', 'X2'])

print('Empirical correlation matrix:')
print(df2.corr().round(3))

In [ ]:
sns.jointplot(x='X1', y='X2', data=df2, kind='scatter', alpha=0.4)
plt.suptitle('2-Variable Cholesky Sample (target ρ = 0.7)', y=1.02)
plt.show()

## Section 2 — Generalising to N Variables

The same function scales directly to any number of variables. Below we generate three correlated variables and inspect all pairwise relationships.

In [ ]:
mean_3d = [0, 0, 0]
cov_3d  = np.array([[1.00,  0.70,  0.40],
                    [0.70,  1.00,  0.50],
                    [0.40,  0.50,  1.00]])

samples_3d = generate_correlated_variables(mean_3d, cov_3d, num_samples=1000)
df3 = pd.DataFrame(samples_3d, columns=['X1', 'X2', 'X3'])

print('Empirical correlation matrix:')
print(df3.corr().round(3))

In [ ]:
sns.pairplot(df3, plot_kws={'alpha': 0.3})
plt.suptitle('3-Variable Cholesky Sample', y=1.02)
plt.show()

## Section 3 — Parametric Case: User-Defined Correlation Matrix

In practice the analyst specifies the correlation matrix from domain knowledge or estimates it from a small sample. The covariance matrix must be **positive definite** for Cholesky decomposition to exist. The helper below validates this before attempting the decomposition.

In [ ]:
def is_positive_definite(matrix):
    """Return True if all eigenvalues are strictly positive."""
    eigenvalues = np.linalg.eigvalsh(matrix)
    return bool(np.all(eigenvalues > 0))


def correlation_to_covariance(corr_matrix, std_devs):
    """Convert a correlation matrix and per-variable std devs to a covariance matrix."""
    D = np.diag(std_devs)
    return D @ corr_matrix @ D

In [ ]:
# --- User-defined inputs ---
variable_names = ['Income', 'Expenses', 'Savings']
means          = [3000, 2000, 500]      # target means
std_devs       = [800,  600,  300]      # target standard deviations

# Correlation matrix (symmetric, ones on diagonal)
corr_matrix = np.array([[1.00,  0.65,  0.30],
                         [0.65,  1.00,  0.10],
                         [0.30,  0.10,  1.00]])

if not is_positive_definite(corr_matrix):
    raise ValueError('Correlation matrix is not positive definite — check your inputs.')

cov_matrix = correlation_to_covariance(corr_matrix, std_devs)

samples = generate_correlated_variables(means, cov_matrix, num_samples=2000)
df_user = pd.DataFrame(samples, columns=variable_names)

print('Descriptive statistics:')
print(df_user.describe().round(1))
print('\nEmpirical correlation matrix:')
print(df_user.corr().round(3))

In [ ]:
sns.pairplot(df_user, plot_kws={'alpha': 0.3})
plt.suptitle('User-Defined Parametric Cholesky Sample', y=1.02)
plt.show()

## Section 4 — Full Pipeline: Distribution Fitting + Validation

When real data is available (even a small sample), we can:
1. **Fit** the best-matching marginal distribution to each variable using the `fitter` library.
2. **Generate** uniform samples and transform them through the fitted CDFs (quantile transformation) to produce correctly distributed marginals.
3. **Inject correlations** via Cholesky decomposition.
4. **Validate** preservation of distributions (KS test) and correlations.

Below we simulate a small real dataset and run the full pipeline.

In [ ]:
np.random.seed(42)

# Simulate a small 'real' dataset (200 records) — replace with pd.read_csv() for actual data
n_real = 200
real_data = pd.DataFrame({
    'Age':    np.random.normal(45, 10, n_real).clip(18, 80).astype(int),
    'Income': np.random.lognormal(8.0, 0.5, n_real),
    'Score':  np.random.beta(2, 5, n_real) * 100,
})

print('Real data sample (first 5 rows):')
print(real_data.head())
print(f'\nShape: {real_data.shape}')

In [ ]:
# Step 1: Fit best distributions to each variable
fitted_distributions = {}
for col in real_data.columns:
    f = Fitter(real_data[col], distributions=['norm', 'lognorm', 'beta', 'gamma', 'expon'])
    f.fit()
    best = f.get_best(method='sumsquare_error')
    fitted_distributions[col] = best
    print(f'{col}: best fit = {list(best.keys())[0]}, params = {list(best.values())[0]}')

In [ ]:
# Step 2: Compute empirical correlation matrix from real data
empirical_corr = real_data.corr().values
print('Empirical correlation matrix:')
print(pd.DataFrame(empirical_corr, index=real_data.columns, columns=real_data.columns).round(3))

In [ ]:
# Step 3: Generate correlated standard-normal samples via Cholesky
n_synthetic = 1000

if not is_positive_definite(empirical_corr):
    # Add small ridge to ensure positive definiteness
    empirical_corr += np.eye(len(real_data.columns)) * 1e-6

L = np.linalg.cholesky(empirical_corr)
z = np.random.normal(size=(n_synthetic, len(real_data.columns)))
correlated_z = z @ L.T  # correlated standard-normal samples

# Convert to uniform [0,1] via the normal CDF
uniform_samples = stats.norm.cdf(correlated_z)

In [ ]:
# Step 4: Transform each column using the quantile function of the fitted distribution
dist_map = {
    'norm':    stats.norm,
    'lognorm': stats.lognorm,
    'beta':    stats.beta,
    'gamma':   stats.gamma,
    'expon':   stats.expon,
}

synthetic_cols = {}
for i, col in enumerate(real_data.columns):
    dist_name = list(fitted_distributions[col].keys())[0]
    params    = list(fitted_distributions[col].values())[0]
    dist_obj  = dist_map[dist_name]
    synthetic_cols[col] = dist_obj.ppf(uniform_samples[:, i], **params)

synthetic_data = pd.DataFrame(synthetic_cols)
print('Synthetic data sample (first 5 rows):')
print(synthetic_data.head())

In [ ]:
# Step 5: Validate — KS test for distribution preservation
print('KS Test (null hypothesis: synthetic and real follow the same distribution)\n')
for col in real_data.columns:
    stat, p = stats.ks_2samp(real_data[col], synthetic_data[col])
    result = 'PASS' if p > 0.05 else 'FAIL'
    print(f'  {col:10s}  KS stat={stat:.4f}  p={p:.4f}  [{result}]')

In [ ]:
# Step 6: Validate — correlation preservation
print('Target correlation matrix (empirical from real data):')
print(real_data.corr().round(3))
print('\nSynthetic data correlation matrix:')
print(synthetic_data.corr().round(3))

In [ ]:
# Step 7: Visual comparison
fig, axes = plt.subplots(1, len(real_data.columns), figsize=(15, 4))
for ax, col in zip(axes, real_data.columns):
    ax.hist(real_data[col],      bins=30, alpha=0.5, label='Real',      density=True)
    ax.hist(synthetic_data[col], bins=30, alpha=0.5, label='Synthetic', density=True)
    ax.set_title(col)
    ax.legend()
plt.suptitle('Real vs Synthetic Marginal Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated four levels of the Cholesky-based synthesis pipeline:

| Section | What was shown |
|---|---|
| 1 — Core concept | 2-variable Cholesky with a fixed covariance matrix |
| 2 — N variables | 3-variable generalisation with pairplot validation |
| 3 — Parametric | User-specified correlation matrix with positive-definiteness check |
| 4 — Full pipeline | Distribution fitting → Cholesky → quantile transform → KS-test validation |

The method preserves both marginal distributions and inter-variable correlations deterministically — without any machine learning component.

**Related publications:**
- Marchev, A., Marchev, V. (2024). *Automated Algorithm for Multi-variate Data Synthesis with Cholesky Decomposition*. ICACS 2023, ACM. DOI: 10.1145/3631908.3631909
- Marchev, A., Marchev, V. (2022). *Synthesizing multi-dimensional personal data sets*. AIP Conference Proceedings. DOI: 10.1063/5.0100615